# Do the mice switch between strategy states?

A **GLM-HMM** (Ashwood et al. 2022, *Nat Neurosci*) and its mixture-of-agents relative (Venditto,
Miller, Brody & Daw 2024, *eLife*) ask whether a mouse moves between strategies over time. Hidden
states with sticky transitions each carry their own weights on the signals of the strategies we
compare: the value of a kernel + forgetting Q-learner, the WSLS (Magnitude) stay and switch signals,
a choice kernel (habit) and a plus-port bias. A mixture over time could explain a flat choice
kernel and the "draw" between the strategy models.

Section 1 runs the method on **simulated agents with known switching** (and on agents that do not
switch, which must choose one state); section 2 selects the real data; section 3 runs the same
simulations on the mice's own problems; section 4 analyzes the mice, with the null of the gain
taken from replays of each mouse's fitted static Q-learner.

In [ ]:
# ruff: noqa: E402  (the repository root is put on sys.path first)
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/behavior_import").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.agents import drifting_simulation as sim
from src.agents import kernel_q_agent
from src.behavior_analysis.glm_hmm import glm_hmm_recovery as recovery
from src.behavior_analysis.data import problem_selection
from src.behavior_visualization.glm_hmm import plot_glm_hmm as glm_plots
from src.pipelines.choice_logit import choice_logit_values as logit_pipe
from src.pipelines.glm_hmm import glm_hmm as pipe
from src.pipelines.switching import switch_regression as switch
from src.settings import get_unchecked_settings
from tests.strategy_fixtures import HABIT, KERNEL_Q, layouts


def show(base):
    # figures are saved (pdf, png, svg, caption) by the plot functions; display the png here
    display(Image(filename=f"{base}.png"))

In [ ]:
# ---- parameters (edit here) ------------------------------------------------------------
COHORTS = [1, 3]  # cohorts whose drifting problems are pooled in the real-data section
DATA_ROOT = None  # None: BANDIT_DATA_ROOT. Loading is cache-first and never writes to the share.
OUTPUT_DIR = REPO_ROOT / "results" / "figures" / "glm-hmm"

# The last drifting problem kept per cohort (None: all); as the rank-signature notebook.
MAX_PROBLEM_BY_COHORT = {1: 7, 3: None}

# The analysis protocol: 1 to 4 states, 10 EM starts (screened for 12 iterations, the best 2
# continue), 5 folds of sessions. Replays and simulations score 1 to 3 states.
CONFIG = pipe.Config()
SEED = 20261112
JOBS = 2  # parallel workers
REPLICATES = 12  # replays of each mouse by its fitted static agent (the null of the gain)
KERNEL_REPLICATES = 3  # of those, the first this many also get the choice kernel by state
SIM_MICE = ["MY_174_LR", "MY_175_L", "MY_174_R"]  # schedules and parameters of section 3
CACHE_DIR = get_unchecked_settings().cache_dir / "glm_hmm"  # disposable

# Section 1: toy schedules (no data needed): problems x sessions x trials per session.
TOY = dict(n_problems=8, sessions=6, trials=80)
TOY_CONFIG = pipe.Config(
    n_states=(1, 2, 3), replay_states=(1, 2, 3), n_restarts=6, n_folds=5, agent_starts=2
)

## 1. Simulated agents with a known answer

A toy mouse (random schedules, the parameters of a typical fitted mouse) is simulated as five agents.
**Switching** agents have three hidden states, each a strategy written as weights on the signals: a
value learner, WSLS (Magnitude) and a habit/biased agent (strong choice kernel and plus-port bias).
Transitions are sticky with a mean dwell of 300 (slow) or 50 (fast) trials, and the state is redrawn
at every session start. **No-switching** controls are the fitted kernel + forgetting Q-learner, the
fitted WSLS (Habit) and a stationary blend (the mean of the three strategies' weights on every
trial). A good GLM-HMM must select one state for the controls, and recover the weights, states and
dwell times of the switching agents.

In [ ]:
from src.behavior_analysis.data.drifting_trial_table import drifting_trial_table

toy_agents = pipe.Agents(kq=KERNEL_Q, habit=HABIT)
toy_rules = sim.TaskRules(drift_rate=0.1, block_lengths=np.arange(30, 36))
toy_layouts = layouts(TOY["n_problems"], TOY["sessions"], TOY["trials"], 0)
toy_trials = drifting_trial_table(
    sim.simulate_policy_mouse(
        lambda layout: sim.KernelQPolicy(KERNEL_Q, "magnitude", layout),
        toy_layouts,
        toy_rules,
        np.random.default_rng(0),
        "toy",
    )
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    toy = [
        pipe.recover_scenario(toy_trials, toy_agents, s, toy_rules, TOY_CONFIG, SEED)
        for s in pipe.SCENARIOS
    ]
rows = []
for r in toy:
    rows.append(
        dict(
            scenario=r.scenario,
            selected_states=r.readouts["selected_k"],
            gain_k2=r.readouts["gain_k2"],
            gain_k3=r.readouts["gain_k3"],
            state_accuracy=r.accuracy,
            weight_corr=(
                recovery.weight_summary(r.weights)["correlation"] if len(r.weights) else np.nan
            ),
        )
    )
display(pd.DataFrame(rows).round(3))

**Reading the toy recovery.** A three-state switching agent is recovered (selected K=3, accuracy 0.98 slow and 0.88 fast, weight correlation about 0.97). The kernel-Q and blend controls select K=1. The static WSLS (Habit) control selects K=2 with a small gain (0.003): the state model absorbs a rule the signals do not span exactly.

## 2. Data selection

Cohort 1 drifting problems 1 to 7 and all of cohort 3, as in the rank-signature notebook
(`problem_selection.select_problems`; the dropped trials are printed).

In [ ]:
trials = logit_pipe.load_drifting_trial_table(COHORTS, DATA_ROOT)
selected = problem_selection.select_problems(trials, MAX_PROBLEM_BY_COHORT)
print("trials before / after the selection:", len(trials), "/", len(selected))
display(problem_selection.dropped_table(trials, selected))
counts = selected.groupby("mouse").agg(
    n_trials=("trial", "size"),
    n_problems=("problem", "nunique"),
    n_sessions=("session", "nunique"),
)
display(counts)
mice = sorted(counts.index)
colors = glm_plots.colors_of([pd.DataFrame({"mouse": mice})])
rules = switch.rules_by_mouse(selected, mice, None)

**Fitted static agents.** Each mouse's kernel + forgetting Q-learner and WSLS (Habit) are fitted to all
of its selected trials (5 random starts). The Q-learner's learning rates, forgetting and choice-kernel rate
make the value and habit **signals**, so every number of states sees the same signals; the fitted
inverse temperature and kernel weight are not used. The same rates make the signals of the simulations
and the replays, so a mouse and its null are analyzed alike. (The signals therefore use rates fitted
to all the mouse's data, while the static models of the comparison below are refitted on the
training folds.)

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    agents = pipe.fit_cohort_agents(selected, n_jobs=JOBS, cache_dir=CACHE_DIR / "agents")
display(
    pd.DataFrame(
        {m: dict(zip(kernel_q_agent.PARAMETER_NAMES, a.kq, strict=True)) for m, a in agents.items()}
    ).T.round(3)
)

## 3. The same simulations on the mice's problems

Three mice with the most trials supply schedules and parameters. Each of the five agents plays that
mouse's problems and is analyzed like a mouse (1 to 4 states, 5 folds of sessions).

In [ ]:
from joblib import Memory, Parallel, delayed

recover = Memory(CACHE_DIR / "recovery", verbose=0).cache(pipe.recover_scenario)
by_mouse = {m: selected[selected["mouse"] == m].reset_index(drop=True) for m in SIM_MICE}
jobs = [(m, s) for m in SIM_MICE for s in pipe.SCENARIOS]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    recoveries = Parallel(n_jobs=JOBS)(
        delayed(recover)(by_mouse[m], agents[m], s, rules[m], CONFIG, SEED, pipe.code_version())
        for m, s in jobs
    )
table = pd.DataFrame(
    [
        dict(mouse=r.mouse, scenario=r.scenario, selected_states=r.readouts["selected_k"],
             accuracy=r.accuracy, **{k: v for k, v in r.readouts.items() if k != "selected_k"})
        for r in recoveries
    ]
)
display(table.round(4))

**Reading the recovery.** On three mice's schedules, slow switching selects K=3 (accuracy 0.96 to 0.99, gain 0.07 to 0.084) and fast switching K=3 (accuracy 0.86 to 0.88, gain 0.045 to 0.051). Static kernel-Q and blend select K=1 (gain about -0.001 to -0.003). Static WSLS (Habit) gives false positives (K=3, 2, 2; gain 0.003 to 0.007), about ten times below genuine switching.

## 4. The mice

Per mouse: held-out gain of 1 to 4 states, the number selected, the static kernel Q-learner and WSLS
(Habit) on the same folds, and the fit at the selected number (at least 2) described by its weights,
occupancy, dwell times, state probability by trial in session, by problem and around block changes,
and the choice kernel within each state. The null of the gain is `REPLICATES` replays per mouse of
its fitted kernel Q-learner, analyzed identically (1 to 3 states).

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    results = pipe.analyze_mice(selected, agents, CONFIG, SEED, n_jobs=JOBS,
                                cache_dir=CACHE_DIR / "mice")
summary = pd.DataFrame(
    [
        dict(mouse=m, trials=r.n_trials, sessions=r.n_sessions, selected_states=r.selected_k,
             **{k: round(v, 4) for k, v in r.readouts.items()},
             **{f"{row.model} nats": round(row.nats, 4) for row in r.static_levels.itertuples()},
             one_state_nats=round(r.table.set_index("n_states").loc[1, "nats"], 4))
        for m, r in results.items()
    ]
).set_index("mouse")
display(summary)

In [ ]:
ks = {m: r.summary_k for m, r in results.items()}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    null = pipe.replay_source(
        selected, agents, CONFIG, REPLICATES, rules=None, seed=SEED, kernel_states=ks,
        kernel_replicates=KERNEL_REPLICATES, n_jobs=JOBS, cache_dir=CACHE_DIR / "replays",
    )
observed = pd.DataFrame(
    [dict(mouse=m, **{k: r.readouts[k] for k in ("gain_k2", "gain_k3", "best_gain")})
     for m, r in results.items()]
)
calibration = pipe.calibrate_gains(observed, null)
display(calibration.table[["mouse", "readout", "observed", "null_mean", "null_sd", "z",
                           "p_empirical"]].round(4))
display(calibration.summary.round(4))
print("replays selecting more than one state:", int((null["selected_k"] > 1).sum()), "of", len(null))

**Reading the calibration.** Replays of the fitted kernel-Q agent have negative mean gains (every mouse) and only 2 of 168 select more than one state. Against that null 9 of 14 mice have z above 2 for the 2-state gain (Wilcoxon p=0.0004 across mice; with 12 replays the smallest possible empirical p is 0.077). The gains are small (0.004 to 0.022 nats per trial) and similar to the static WSLS (Habit) false positives, so the test says the kernel-Q agent is not the whole story, not that the mice switch.

In [ ]:
rows = []
for m, r in results.items():
    w = r.weights[r.weights["n_states"] == r.summary_k].pivot(
        index="state", columns="feature", values="weight")
    occ = r.occupancy.set_index("state")
    for z in w.index:
        rows.append(dict(mouse=m, summary_k=r.summary_k, state=z + 1,
                         occupancy=occ.loc[z, "occupancy"], dwell=occ.loc[z, "dwell"],
                         **w.loc[z].round(2).to_dict()))
display(pd.DataFrame(rows).round(2).set_index(["mouse", "state"]))
kern = pd.concat({m: r.kernel for m, r in results.items()}, names=["mouse"]).reset_index(level=0)
print(kern.pivot_table(index=["mouse", "state"], columns="lag", values="weight").round(2))

In [ ]:
data = glm_plots.FigureData(
    mice=results, null=null, calibration=calibration, recovery=recoveries,
    replicates=REPLICATES, colors=colors,
)
glm_plots.plot_all(data, OUTPUT_DIR)

In [ ]:
show(OUTPUT_DIR / glm_plots.MODEL_SELECTION)

**Reading GLM-HMM Model Selection.** Left: gain over one state by K (gray: the replay band); the static models sit below one state. Middle: z of the gain against replays. Right: the selected K is 2 for 7 mice, 3 for 2 and 1 for 5 (MY_175_N, MY_66_R, MY_71_N, MY_72_N, MY_72_R). Gains rarely improve beyond K=2.

In [ ]:
show(OUTPUT_DIR / glm_plots.STATE_WEIGHTS)

**Reading GLM-HMM State Weights.** At each mouse's summary K, state 1 is the state with the largest Q weight (about 1.0 on average against about 0.3 in state 2). States differ mainly in value use and plus-port bias (state 2 has the larger bias), not in a clean value / WSLS / habit split. Hollow dots are mice with one selected state (weights from their summary fit).

In [ ]:
show(OUTPUT_DIR / glm_plots.STATE_OCCUPANCY)

**Reading GLM-HMM State Occupancy.** Occupancy is about 0.5 per state with dwell times of about 10 to 100 trials (sessions are about 70 to 100 trials, so many states last under a session). The state-1 (value) probability rises across problems and does not change around block changes. The late-session curves rest on few sessions.

In [ ]:
show(OUTPUT_DIR / glm_plots.SIMULATED_RECOVERY)

**Reading GLM-HMM Simulated Recovery.** Switching agents select 3 states and recover the weights (diagonal) and state sequence; fast switching is recovered less well, and the fitted dwell times are noisy. Only static WSLS (Habit) shows a false-positive gain. The empty accuracy panels for the controls are by design.

In [ ]:
show(OUTPUT_DIR / glm_plots.KERNEL_BY_STATE)

**Reading Choice Kernel by State.** State-specific kernels are noisy (6 to 8 mice qualify) and not consistently flatter or sharper than the pooled kernel (flatness median under 1 in both states); the static agents' kernels show the same spread. So the data give no evidence that the kernel shape depends on the state. The ratio is unstable when lag 1 is near zero (symmetric log axis).